In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import os
import pandas as pd
import numpy as np

In [3]:
target_files = [
    "CRMLSSold202512.csv",
    "CRMLSSold202601.csv",
    "CRMLSSold202602.csv",
    "CRMLSSold202603.csv",
    "CRMLSSold202604.csv",
    "CRMLSSold202605.csv"
]

data_path = "../DS intern file" 
df_list = []

for file_name in target_files:
    file_key = os.path.join(data_path, file_name)
    if os.path.exists(file_key):
        temp_df = pd.read_csv(file_key, low_memory=False)
        df_list.append(temp_df)
    elif os.path.exists(file_name): # 현재 경로에 있는 경우 대비
        temp_df = pd.read_csv(file_name, low_memory=False)
        df_list.append(temp_df)

if not df_list:
    # 만약 기존에 만들어둔 cleaned_crmls_sold.csv를 활용하는 경우
    df = pd.read_csv("cleaned_crmls_sold.csv")
else:
    df = pd.concat(df_list, ignore_index=True)

filtered_df = df[
    (df['PropertyType'] == 'Residential') & 
    (df['PropertySubType'] == 'SingleFamilyResidence')
].copy()

In [8]:
def find_column(df, keywords, default_name):
    for col in df.columns:
        if any(kw.lower() in col.lower() for kw in keywords):
            return col
    return default_name

price_col = find_column(filtered_df, ['ClosePrice'], 'ClosePrice')
living_col = find_column(filtered_df, ['LivingArea'], 'LivingArea')
beds_col = find_column(filtered_df, ['BedroomsTotal', 'MainLevelBedrooms', 'Bedrooms'], 'Bedrooms')
baths_col = find_column(filtered_df, ['BathroomsTotalInteger', 'BathroomsFull', 'Bathrooms'], 'Bathrooms')
lot_col = find_column(filtered_df, ['LotSizeSquareFeet', 'LotSizeArea', 'LotSizeAcres', 'LotSize'], 'LotSize')
year_col = find_column(filtered_df, ['YearBuilt'], 'YearBuilt')

# 🌟 [핵심] 위도/경도 컬럼 반드시 감지!
lat_col = find_column(filtered_df, ['Latitude', 'Lat'], 'Latitude')
lon_col = find_column(filtered_df, ['Longitude', 'Lon', 'Lng'], 'Longitude')

# 날짜 정렬 및 Target Log 변환
if 'CloseDate' in filtered_df.columns:
    filtered_df['CloseDate'] = pd.to_datetime(filtered_df['CloseDate'])
    filtered_df = filtered_df.sort_values('CloseDate').reset_index(drop=True)

filtered_df['target_log_price'] = np.log1p(filtered_df[price_col])

# Train / Test Split 플래그 생성 (상위 20%를 Test로 설정)
split_idx = int(len(filtered_df) * 0.8)
filtered_df['is_test'] = 0
filtered_df.iloc[split_idx:, filtered_df.columns.get_loc('is_test')] = 1


In [9]:
filtered_df['bed_bath_ratio'] = filtered_df[beds_col] / (filtered_df[baths_col] + 1)
filtered_df['property_age'] = 2026 - filtered_df[year_col].fillna(2000)
filtered_df['living_to_lot_ratio'] = filtered_df[living_col] - filtered_df[lot_col]

# 2) 🌟 School District GeoJSON 데이터 공간 결합 (Spatial Join)
geojson_file = "California_School_District_Areas_2024-25.geojson"

try:
    gdf_properties = gpd.GeoDataFrame(
        filtered_df, 
        geometry=gpd.points_from_xy(filtered_df[lon_col], filtered_df[lat_col]),
        crs="EPSG:4326"
    )

    school_districts = gpd.read_file(geojson_file)
    if gdf_properties.crs != school_districts.crs:
        school_districts = school_districts.to_crs(gdf_properties.crs)

    joined_gdf = gpd.sjoin(gdf_properties, school_districts, how="left", predicate="intersects")
    school_col = [c for c in school_districts.columns if 'NAME' in c.upper() or 'DISTRICT' in c.upper()][0]
    
    filtered_df['school_district'] = joined_gdf[school_col].fillna('Unknown')
    print("  -> 🎉 CA School District Layer 공간 조인(Spatial Join) 성공!")
except Exception as e:
    print(f"  ⚠️ Spatial Join 실행 중 참고: {e}")
    filtered_df['school_district'] = 'Unknown'

  ⚠️ Spatial Join 실행 중 참고: name 'gpd' is not defined


In [28]:
def split_data_by_window(df, test_start_date_str, X_months=3):

   # data processing
    df['CloseDate'] = pd.to_datetime(df['CloseDate'], errors='coerce')
    df = df.dropna(subset=['CloseDate']).copy()
    
   # change to date
    test_start_date = pd.to_datetime(test_start_date_str)
    
    # Test end date
    test_end_date = test_start_date + pd.DateOffset(months=1)
    
    # train start date
    train_start_date = test_start_date - pd.DateOffset(months=X_months)
    
    # data slicing
    train_set = df[(df['CloseDate'] >= train_start_date) & (df['CloseDate'] < test_start_date)].copy()
    test_set = df[(df['CloseDate'] >= test_start_date) & (df['CloseDate'] < test_end_date)].copy()
    
    print(f"  - Train: {train_start_date.strftime('%Y-%m-%d')} ~ {test_start_date.strftime('%Y-%m-%d')} ({len(train_set)})")
    print(f"  - Test: {test_start_date.strftime('%Y-%m-%d')} ~ {test_end_date.strftime('%Y-%m-%d')} ({len(test_set)})")
    
    return train_set, test_set

In [30]:
test_target_date = "2026-05-01" 
X_window_size = 3 

train_data, test_data = split_data_by_window(filtered_df, test_target_date, X_months=X_window_size)

X_train_raw = train_data[numeric_features + categorical_features]
X_test_raw = test_data[numeric_features + categorical_features]

y_train = np.log1p(train_data[price_col])
y_test = np.log1p(test_data[price_col])


  - Train: 2026-02-01 ~ 2026-05-01 (31758건)
  - Test: 2026-05-01 ~ 2026-06-01 (12024건)


In [32]:
# fit and transform 
X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

# replace column name
cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
encoded_cat_cols = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + encoded_cat_cols

# Make dataframe
X_train_df = pd.DataFrame(X_train_processed, columns=all_features, index=train_data.index)
X_train_df['target_log_price'] = y_train
X_train_df['is_test'] = 0 

X_test_df = pd.DataFrame(X_test_processed, columns=all_features, index=test_data.index)
X_test_df['target_log_price'] = y_test
X_test_df['is_test'] = 1   

# 6. 병합 후 클렌징된 CSV로 영구 저장
cleaned_csv_df = pd.concat([X_train_df, X_test_df], axis=0)
cleaned_csv_df.to_csv("cleaned_crmls_sold.csv", index=False)

print(cleaned_csv_df)

       LivingArea  BedroomsTotal  BathroomsTotalInteger  LotSizeAcres  \
17945   -0.093094       0.526197              -0.568203     -0.017498   
17946   -0.538801      -1.546351              -0.568203     -0.018761   
17947   -0.804492       0.526197              -0.568203     -0.018789   
17948    1.488539       2.598745               2.073429     -0.018753   
17949    0.036864      -0.510077               0.312341     -0.018959   
...           ...            ...                    ...           ...   
61722   -0.548428      -0.510077              -0.568203     -0.018784   
61723   -0.627365      -0.510077              -0.568203     -0.018820   
61724   -0.959479      -0.510077              -0.568203     -0.018830   
61725    0.302555      -0.510077               0.312341      0.019702   
61726   -0.296213      -0.510077              -0.568203     -0.018788   

       PropertyType_Residential  PropertySubType_SingleFamilyResidence  \
17945                       1.0                  